In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


ModuleNotFoundError: No module named 'torch'

In [ ]:
import torch
import torch.nn as nn
import timm

# 1. Define the exact same model architecture used for training
class HybridResNetViT(nn.Module):
    def __init__(self):
        super(HybridResNetViT, self).__init__()
        # Use pretrained=False as we are loading weights afterwards
        self.resnet = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(2816, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        resnet_features = self.resnet(x)
        vit_features = self.vit(x)
        fused_features = torch.cat((resnet_features, vit_features), dim=1)
        return self.classifier(fused_features)

# Set device (CPU or GPU based on availability)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model
loaded_model = HybridResNetViT().to(device)

# Load the saved state dictionary
# map_location ensures it loads correctly whether you saved from GPU or CPU
loaded_model.load_state_dict(torch.load('/kaggle/input/models/srijaymodel/hyb/pytorch/default/1/best_hybrid_pneumonia_model.pth', map_location=device))

# Set the model to evaluation mode
loaded_model.eval()

print("Model 'best_hybrid_pneumonia_model.pth' loaded successfully!")
print("You can now use 'loaded_model' for inference.")

In [ ]:
# No API keys needed, just point to the local Kaggle directories!
MODEL_WEIGHTS_PATH = '/kaggle/input/models/srijaymodel/hyb/pytorch/default/1/best_hybrid_pneumonia_model.pth'
DATASET_PATH = '/kaggle/input/datasets/yusufmurtaza01/chest-xray-pneumonia-balanced-dataset'

Success! kaggle.json has been created and secured.


In [ ]:
!pip install -q timm scikit-learn tqdm

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from tqdm import tqdm
import copy

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define image transformations
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        normalize
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize
    ]),
}

# The path based on our extraction command
data_dir = '/kaggle/input/datasets/yusufmurtaza01/chest-xray-pneumonia-balanced-dataset'

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# Batch size is set to 16 to prevent Out-Of-Memory errors with two large models
dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=(x == 'train'), num_workers=2)
               for x in ['train', 'val', 'test']}

print(f"Classes: {image_datasets['train'].classes}")


Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:37<00:00, 281MB/s]
100% 2.29G/2.29G [00:37<00:00, 65.0MB/s]


In [ ]:
import torch
import torch.nn as nn
import timm

# 1. Build the "Blueprint" (Notice pretrained=False)
class HybridResNetViT(nn.Module):
    def __init__(self):
        super(HybridResNetViT, self).__init__()
        self.resnet = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(2816, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        resnet_features = self.resnet(x)
        vit_features = self.vit(x)
        fused_features = torch.cat((resnet_features, vit_features), dim=1)
        return self.classifier(fused_features)

# 2. Construct the empty "House" on the CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridResNetViT().to(device)

# 3. Move the "Furniture" (your .pth weights) into the house
MODEL_WEIGHTS_PATH = '/kaggle/input/models/srijaymodel/hyb/pytorch/default/1/best_hybrid_pneumonia_model.pth'
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))

# 4. Lock the doors for evaluation
model.eval()
print("Model built and custom weights loaded successfully!")

Using device: cuda
Classes: ['NORMAL', 'PNEUMONIA']


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1)

num_epochs = 10
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs} | ' + '-' * 20)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0
        total_samples = 0

        # Iterate over data using a progress bar
        for inputs, labels in tqdm(dataloaders[phase], desc=phase):
            inputs = inputs.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                preds = torch.sigmoid(outputs) > 0.5

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects.double() / total_samples

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        # Save the best model based on validation accuracy
        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, 'best_hybrid_pneumonia_model_dataset2.pth')
            print(">>> Best model saved!")

        if phase == 'val':
            scheduler.step(epoch_loss)

print('Training complete. Best Val Acc: {:4f}'.format(best_acc))

# Load best model weights for the final test evaluation
model.load_state_dict(best_model_wts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from tqdm import tqdm

model.eval()
test_preds = []
test_labels = []
test_probs = []

# 1. Run Inference
with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc='Testing'):
        inputs = inputs.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        preds = probs > 0.5

        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_labels = np.array(test_labels)
test_preds = np.array(test_preds)
test_probs = np.array(test_probs)

# 2. Calculate Advanced Metrics
cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print("\n" + "="*30)
print(" 🏥 ADVANCED CLINICAL METRICS ")
print("="*30)
print(f"Accuracy:             {accuracy_score(test_labels, test_preds):.4f}")
print(f"Precision:            {precision_score(test_labels, test_preds):.4f}")
print(f"Recall (Sensitivity): {recall_score(test_labels, test_preds):.4f}  <- Crucial for catching sick patients")
print(f"Specificity:          {specificity:.4f}  <- Crucial for clearing healthy patients")
print(f"F1-Score:             {f1_score(test_labels, test_preds):.4f}")
print(f"ROC-AUC:              {roc_auc_score(test_labels, test_probs):.4f}")
print("="*30)

# 3. Plot the Confusion Matrix
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Pneumonia'], 
            yticklabels=['Normal', 'Pneumonia'],
            annot_kws={"size": 14})
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('Actual Diagnosis', fontsize=12)
plt.xlabel('AI Prediction', fontsize=12)

# 4. Plot the ROC Curve
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
plt.subplot(1, 2, 2)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_score(test_labels, test_probs):.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC)', fontsize=16)
plt.legend(loc="lower right")

plt.tight_layout()
plt.show()

Epoch 1/10 | --------------------


train: 100%|██████████| 326/326 [04:06<00:00,  1.32it/s]


Train Loss: 0.2593 Acc: 0.8877


val: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


Val Loss: 0.3970 Acc: 0.7500
>>> Best model saved!
Epoch 2/10 | --------------------


...

train: 100%|██████████| 326/326 [04:13<00:00,  1.29it/s]


Train Loss: 0.0508 Acc: 0.9816


val: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Val Loss: 0.8945 Acc: 0.5625
Training complete. Best Val Acc: 0.937500


<All keys matched successfully>

In [13]:
!pip install -q gradio

In [14]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import timm

# 1. Define the exact same model architecture so we can load the weights
class HybridResNetViT(nn.Module):
    def __init__(self):
        super(HybridResNetViT, self).__init__()
        self.resnet = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(2816, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        resnet_features = self.resnet(x)
        vit_features = self.vit(x)
        fused_features = torch.cat((resnet_features, vit_features), dim=1)
        return self.classifier(fused_features)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model and load your saved weights
model = HybridResNetViT().to(device)
# Ensure the path matches where you saved the weights in the training loop
model.load_state_dict(torch.load('best_hybrid_pneumonia_model.pth', map_location=device))
model.eval()

# 2. Define the image transformations (must match validation/test transforms exactly)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Define the prediction function that Gradio will call
def predict_pneumonia(image):
    # Convert Gradio image (PIL) to RGB (in case it's grayscale)
    image = image.convert('RGB')

    # Preprocess and add batch dimension [1, 3, 224, 224]
    input_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probability = torch.sigmoid(output).item()

    # Format the output for the interface
    if probability > 0.5:
        result = f"PNEUMONIA DETECTED 🚨\nConfidence: {probability * 100:.2f}%"
    else:
        result = f"NORMAL (No Pneumonia) ✅\nConfidence: {(1 - probability) * 100:.2f}%"

    return result

# 4. Build and launch the Gradio Interface
demo = gr.Interface(
    fn=predict_pneumonia,
    inputs=gr.Image(type="pil", label="Upload Chest X-Ray"),
    outputs=gr.Textbox(label="Diagnosis Result", lines=2),
    title="Hybrid Deep Learning Pneumonia Detector",
    description="Upload a chest X-ray image. The system uses a combined ResNet-50 and Vision Transformer (ViT) architecture to detect signs of pneumonia.",
    theme="huggingface",
    allow_flagging="never"
)

# Launch the app (creates a public link you can share)
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1143: UserWarning: Cannot load huggingface. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/huggingface' (Request ID: Root=1-69b174bb-49022a51158d1f7279e8d79a;a3ca402a-71ba-4dc8-8b93-488f7cf610ce)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4ebe5bd29618545e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
